# 🎬 Netflix Content Analysis & Machine Learning
**Author:** Gayathri Kota | Arizona State University  
**Tools:** Python · Pandas · NumPy · Seaborn · Matplotlib · Scikit-learn  
**Dataset:** Netflix Movies and TV Shows (Kaggle) — 8,807 titles

---

## 📌 Project Overview

In this project, I analyze the **Netflix Titles dataset** to uncover content patterns and apply two machine learning methods:

1. **Random Forest Classification** — Predict whether a title is a Movie or a TV Show using metadata features
2. **K-Means Clustering** — Discover natural groupings in the Netflix catalog based on content characteristics

The goal is to practice real-world data cleaning, exploratory analysis, feature engineering, and machine learning on an entertainment dataset.

---

## 📋 Table of Contents
1. [Background & Problem Definition](#1-background)
2. [Imports & Data Loading](#2-imports)
3. [Data Cleaning](#3-cleaning)
4. [Exploratory Data Analysis (EDA)](#4-eda)
5. [Random Forest Classification](#5-random-forest)
6. [K-Means Clustering](#6-kmeans)
7. [Conclusion](#7-conclusion)

---
## 1. Background & Problem Definition <a id='1-background'></a>

### Dataset
The **Netflix Movies and TV Shows** dataset from Kaggle contains **8,807 rows and 12 columns**. Each row represents one title available on Netflix.

| Column | Description | Example |
|--------|-------------|---------|
| `type` | Movie or TV Show | `Movie` |
| `title` | Title name | `"Inception"` |
| `director` | Director name | `"Christopher Nolan"` |
| `cast` | Actors | `"Leonardo DiCaprio"` |
| `country` | Country of origin | `"United States"` |
| `date_added` | When added to Netflix | `"2021-01-01"` |
| `release_year` | Year released | `2010` |
| `rating` | Content rating | `"PG-13"`, `"TV-MA"` |
| `duration` | Length | `"148 min"` or `"2 Seasons"` |
| `listed_in` | Genre(s) | `"Action, Drama"` |
| `description` | Short summary | `"A thief who..."` |

### Research Questions

**1. Classification Problem:**  
Can we predict whether a Netflix title is a **Movie** or **TV Show** using only its metadata (release year, duration, rating, genres)?

**2. Clustering Problem:**  
Can we find **natural groups** of titles based on features like release year, duration, and genre count — without using labels?

### Methods Applied
- 🌲 **Random Forest Classifier** → Supervised learning (needs labels)
- 📍 **K-Means Clustering** → Unsupervised learning (no labels needed)

---
## 2. Imports & Data Loading <a id='2-imports'></a>

### Why these libraries?
| Library | Purpose |
|---------|---------|
| `pandas` | Loading CSVs, data manipulation, groupby, filtering |
| `numpy` | Numerical operations, handling NaN values |
| `matplotlib` / `seaborn` | Creating charts and visualizations |
| `sklearn` | All machine learning — Random Forest, K-Means, PCA, scaling |

In [ ]:
# ── Import all required libraries ──────────────────────────────────────────
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sns.set(style="whitegrid")
print("✅ All libraries imported successfully!")

In [ ]:
# ── Load the dataset ───────────────────────────────────────────────────────
# Make sure netflix_titles.csv is in the same folder as this notebook
df = pd.read_csv("netflix_titles.csv")

print(f"Dataset shape: {df.shape}")  # (rows, columns)
df.head()  # Preview the first 5 rows

In [ ]:
# ── Basic dataset information ──────────────────────────────────────────────
# This tells us:
#   - How many rows and columns exist
#   - What data type each column is
#   - Which columns have missing values and how many

print("Shape of data:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nMissing values per column:\n", df.isna().sum())

---
## 3. Data Cleaning <a id='3-cleaning'></a>

Raw data is rarely ready for machine learning. We need to:
1. Handle missing values
2. Convert text-based columns into numbers (ML only works with numbers)
3. Engineer new features that capture useful information

### Cleaning Steps Overview

| Step | Problem | Solution |
|------|---------|----------|
| Missing `country`, `rating`, `listed_in` | NaN breaks ML models | Fill with `"Unknown"` |
| `duration` is a string like `"90 min"` | ML needs numbers | Extract just the number with `parse_duration()` |
| Missing `duration_num` after extraction | Some entries had no duration | Fill with **median per type** (Movies & TV Shows separately) |
| `num_countries` and `num_genres` | More features = better model | Count items in comma-separated strings |

### Step 1 — Understand the `duration` column first

In [ ]:
# ── Preview the duration column before cleaning ────────────────────────────
# Notice it's a STRING like "90 min" or "2 Seasons" — not a number
# We need to extract just the numeric part for ML

print("Sample duration values:")
print(df['duration'].head(10).tolist())
print("\nData type:", df['duration'].dtype)

### Step 2 — Extract numeric duration with a custom function

**Why a custom function?**  
Movies store duration in **minutes** (e.g., `"148 min"`) while TV Shows store it in **seasons** (e.g., `"3 Seasons"`). A simple split on the first word gives us just the number in both cases.  

We use `df.apply()` to run this function on every row.

In [ ]:
# ── Parse duration: extract number from string ─────────────────────────────
def parse_duration(row):
    """
    Extracts the numeric part from the duration string.
    
    Examples:
        "90 min"    → 90   (minutes for movies)
        "2 Seasons" → 2    (seasons for TV shows)
        NaN          → NaN  (missing values stay NaN)
    
    Why: ML models require numeric input.
    We cannot use "90 min" directly — we need the number 90.
    """
    dur = row['duration']
    if pd.isna(dur):
        return np.nan
    
    num = dur.split()[0]  # take the first word, e.g. "90" from "90 min"
    try:
        return int(num)
    except:
        return np.nan

# Apply this function to every row in the dataframe
df['duration_num'] = df.apply(parse_duration, axis=1)

# Verify the result
print("Before and after parsing:")
df[['type', 'duration', 'duration_num']].head(10)

### Step 3 — Handle missing values

**Why fill with `"Unknown"` instead of dropping?**  
Dropping rows with missing values would remove thousands of titles and shrink our dataset significantly. Replacing with `"Unknown"` keeps all the data while marking that the information was unavailable.

**Why fill `duration_num` with the median per type?**  
A movie's average duration (~90 min) is very different from a TV show's (~2 seasons). Using one overall median would give incorrect estimates. Grouping by `type` first gives a more accurate fill for each category.

In [ ]:
# ── Fill missing values ────────────────────────────────────────────────────

# Fill text columns with "Unknown" so they don't cause errors
df['country']   = df['country'].fillna("Unknown")
df['rating']    = df['rating'].fillna("Unknown")
df['listed_in'] = df['listed_in'].fillna("Unknown")

# Fill missing duration_num using the MEDIAN per type (Movie vs TV Show)
# This is smarter than using one overall median because:
#   - Movie median ≈ 90 minutes
#   - TV Show median ≈ 1-2 seasons
# Mixing them would give wrong estimates
df['duration_num'] = df.groupby('type')['duration_num'] \
                       .transform(lambda x: x.fillna(x.median()))

# Confirm no more missing values in key columns
print("Missing values after cleaning:")
print(df[['country','rating','listed_in','duration_num']].isna().sum())

### Step 4 — Feature Engineering

We create two new columns that capture useful information hidden inside text:
- `num_countries` — how many countries a title is associated with
- `num_genres` — how many genres a title belongs to

**Why are these useful?**  
A title associated with multiple countries or genres might behave differently in our model. These numeric features help the model learn those patterns.

In [ ]:
# ── Feature Engineering ────────────────────────────────────────────────────

# Count how many countries are listed (comma-separated)
# Example: "United States, United Kingdom" → 2
df['num_countries'] = df['country'].apply(lambda x: len(str(x).split(',')))

# Count how many genres are listed (comma-separated)
# Example: "Dramas, International Movies, Thrillers" → 3
df['num_genres'] = df['listed_in'].apply(lambda x: len(str(x).split(',')))

print("New features created:")
df[['country', 'num_countries', 'listed_in', 'num_genres']].head(8)

---
## 4. Exploratory Data Analysis (EDA) <a id='4-eda'></a>

Before building models, we explore the data visually to understand its structure, spot patterns, and decide which features matter most.

**Four questions we answer here:**
1. Are there more Movies or TV Shows on Netflix?
2. When were most titles released?
3. How long are movies vs TV shows?
4. What genres dominate the catalog?

### 4.1 — Movies vs TV Shows

In [ ]:
# ── Chart 1: Count of Movies vs TV Shows ──────────────────────────────────
# This tells us the class balance of our target variable.
# If one class is much larger, our ML model needs to account for that.

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='type', palette='Blues_d')
plt.title("Count of Movies vs TV Shows on Netflix", fontsize=13)
plt.xlabel("")
plt.ylabel("Count")

# Annotate bars with exact counts
for p in plt.gca().patches:
    plt.gca().annotate(f'{int(p.get_height()):,}',
                       (p.get_x() + p.get_width() / 2., p.get_height()),
                       ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

print("\nBreakdown:")
print(df['type'].value_counts())
print(f"\nMovies make up {df['type'].value_counts(normalize=True)['Movie']*100:.1f}% of the dataset")

**Observation:** Movies significantly outnumber TV Shows on Netflix. This class imbalance is important to keep in mind for our Random Forest model — we use `stratify=y` during the train/test split to maintain this ratio in both sets.

### 4.2 — Distribution of Release Year

In [ ]:
# ── Chart 2: Release Year Distribution ────────────────────────────────────
# This shows us WHEN Netflix content was created.
# A spike after 2010 tells us Netflix's catalog is mostly recent content.

plt.figure(figsize=(10, 5))
sns.histplot(df['release_year'], bins=40, kde=False, color='steelblue')
plt.title("Distribution of Release Year", fontsize=13)
plt.xlabel("Release Year")
plt.ylabel("Number of Titles")
plt.axvline(x=2010, color='red', linestyle='--', alpha=0.6, label='Year 2010')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Titles released before 2000: {(df['release_year'] < 2000).sum()}")
print(f"Titles released 2000–2009:   {((df['release_year'] >= 2000) & (df['release_year'] < 2010)).sum()}")
print(f"Titles released 2010+:       {(df['release_year'] >= 2010).sum()}")

**Observation:** The vast majority of Netflix titles were released after 2010. This makes `release_year` a useful feature — older vs. newer titles may have different characteristics (genre trends, duration norms, etc.).

### 4.3 — Duration: Movies vs TV Shows

In [ ]:
# ── Chart 3: Duration Boxplot ──────────────────────────────────────────────
# A boxplot shows the spread and center of a distribution.
# This is especially interesting because:
#   - Movies: duration is in MINUTES (typically 80-120)
#   - TV Shows: duration is in SEASONS (typically 1-3)
# Even though they use different units, the numeric difference
# is exactly what the Random Forest model will learn to use!

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='type', y='duration_num', palette='Set2')
plt.title("Duration by Type (Movies = minutes | TV Shows = seasons)", fontsize=13)
plt.xlabel("")
plt.ylabel("Duration (minutes or seasons)")
plt.tight_layout()
plt.show()

print("\nDuration statistics by type:")
print(df.groupby('type')['duration_num'].describe().round(1))

**Observation:** The difference in `duration_num` between Movies and TV Shows is enormous — movies average ~90 minutes while TV shows average ~1-2 seasons. This is why `duration_num` becomes the **#1 most important feature** in our Random Forest model later.

### 4.4 — Top 10 Genres

In [ ]:
# ── Chart 4: Top 10 Genres ─────────────────────────────────────────────────
# Netflix genres are stored as comma-separated strings like:
# "Dramas, International Movies, Thrillers"
# We split each one and count individual genre occurrences.

from collections import Counter

genre_counts = Counter()
for x in df['listed_in']:
    for g in str(x).split(','):
        genre_counts[g.strip()] += 1

top_genres = genre_counts.most_common(10)
genres, counts = zip(*top_genres)

plt.figure(figsize=(10, 5))
sns.barplot(x=list(counts), y=list(genres), palette='mako')
plt.title("Top 10 Genres / Categories on Netflix", fontsize=13)
plt.xlabel("Number of Titles")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

print("\nTop 10 genres:")
for g, c in top_genres:
    print(f"  {g}: {c:,} titles")

**Observation:** International Movies and Dramas dominate Netflix's catalog. This reflects Netflix's global expansion strategy. Comedies and Documentaries also rank highly, giving us a diverse genre distribution for our clustering model.

---
## 5. Random Forest Classification <a id='5-random-forest'></a>

### Goal
Predict whether a Netflix title is a **Movie (0)** or **TV Show (1)** using only its metadata.

### Why Random Forest?
| Reason | Explanation |
|--------|-------------|
| Handles mixed data | Works well with numeric + encoded categorical features |
| Robust to noise | Averaging many decision trees reduces overfitting |
| Feature importance | Tells us WHICH features matter most — great for interpretation |
| No scaling needed | Unlike SVM or KNN, Random Forest doesn't require normalized features |

### Feature Engineering for Classification

| Feature | Type | Why included |
|---------|------|-------------|
| `release_year` | Numeric | Older vs newer content differs |
| `duration_num` | Numeric | Most predictive — movies vs TV shows differ drastically |
| `num_countries` | Numeric | Multi-country productions may be different content types |
| `num_genres` | Numeric | Genre breadth varies by type |
| `rating` | Categorical → one-hot encoded | Content rating correlates with type (TV-MA, PG-13, etc.) |

### Step 1 — Prepare Features and Target

In [ ]:
# ── Prepare X (features) and y (target) ───────────────────────────────────

# Encode target: Movie = 0, TV Show = 1
df['type_binary'] = df['type'].map({'Movie': 0, 'TV Show': 1})

# Select feature columns
feature_cols = ['release_year', 'duration_num', 'num_countries', 'num_genres', 'rating']

# One-hot encode the 'rating' column
# Why? Random Forest needs numbers. "TV-MA", "PG-13" are strings.
# pd.get_dummies() converts each unique rating into its own 0/1 column.
# drop_first=True removes one column to avoid multicollinearity.
X = df[feature_cols].copy()
X = pd.get_dummies(X, columns=['rating'], drop_first=True)

y = df['type_binary']

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape:  {y.shape}")
print(f"\nFeature columns ({len(X.columns)} total):")
print(X.columns.tolist())

### Step 2 — Train / Test Split

We split data **75% training / 25% testing**.

- **Training set** → the model learns from this
- **Testing set** → we evaluate the model on data it has NEVER seen

We use `stratify=y` to ensure both sets have the same Movie/TV Show ratio as the full dataset. Without this, the split might accidentally put most TV Shows in one set.

In [ ]:
# ── Train / Test Split ─────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,      # 25% goes to testing
    random_state=42,     # ensures same split every time (reproducibility)
    stratify=y           # keeps Movie/TV Show ratio balanced in both sets
)

print(f"Training set:  {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Testing set:   {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.0f}%)")

print(f"\nClass distribution in training set:")
print(y_train.value_counts().rename({0:'Movie', 1:'TV Show'}))

### Step 3 — Train the Random Forest Model

In [ ]:
# ── Train Random Forest Classifier ────────────────────────────────────────
# n_estimators=200 → builds 200 decision trees (more trees = more stable)
# max_depth=None   → trees can grow as deep as needed
# random_state=42  → reproducibility
# n_jobs=-1        → uses all CPU cores for faster training

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Make predictions on the test set
y_pred = rf.predict(X_test)

print("✅ Model trained successfully!")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Movie', 'TV Show']))

### Step 4 — Confusion Matrix

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────────────
# A confusion matrix shows us:
#   True Positives  → correctly predicted Movies
#   True Negatives  → correctly predicted TV Shows
#   False Positives → predicted Movie but was actually TV Show
#   False Negatives → predicted TV Show but was actually Movie

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=['Predicted Movie', 'Predicted TV Show'],
            yticklabels=['Actual Movie', 'Actual TV Show'])
plt.title("Confusion Matrix — Random Forest", fontsize=13)
plt.tight_layout()
plt.show()

total = cm.sum()
correct = cm[0,0] + cm[1,1]
print(f"Overall accuracy: {correct/total*100:.1f}%")

### Step 5 — Feature Importance

Feature importance tells us **which features the model relied on most** when making predictions. This is one of the most valuable parts of using Random Forest — it gives us interpretability.

In [ ]:
# ── Feature Importance Chart ───────────────────────────────────────────────
# rf.feature_importances_ gives a score between 0 and 1 for each feature.
# Higher score = the model used that feature more to make decisions.
# All scores sum to 1.0.

importances = pd.Series(rf.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(9, 5))
sns.barplot(x=importances_sorted.values, y=importances_sorted.index, palette='viridis')
plt.title("Top 15 Feature Importances — Random Forest", fontsize=13)
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

print("Top 5 most important features:")
for feat, score in importances_sorted.head(5).items():
    print(f"  {feat}: {score:.4f}")

**Key Finding:** `duration_num` is by far the most important feature. This makes complete sense — movies are measured in **minutes** (typically 80–160) while TV shows are measured in **seasons** (typically 1–5). The model learned this pattern immediately.

`release_year` and one-hot encoded `rating` columns also contributed meaningfully.

---
## 6. K-Means Clustering <a id='6-kmeans'></a>

### Goal
Find **natural groups** of Netflix titles without using labels — purely based on content features.

### Why K-Means?
| Reason | Explanation |
|--------|-------------|
| Unsupervised | No labels needed — discovers hidden patterns |
| Simple & interpretable | Each title belongs to exactly one cluster |
| Scalable | Works well on 8,000+ rows |

### Features used for clustering
- `release_year` — when the title was made
- `duration_num` — how long it is
- `num_genres` — how many genres it spans

**⚠️ Important:** We must scale features before K-Means because it uses **Euclidean distance**. Without scaling, `duration_num` (0–300+ min) would dominate over `num_genres` (1–4). StandardScaler puts all features on the same scale.

### Step 1 — Scale the Features

In [ ]:
# ── Prepare and Scale Features for Clustering ─────────────────────────────
cluster_features = df[['release_year', 'duration_num', 'num_genres']].copy()

# StandardScaler transforms each column to have:
#   mean = 0 and standard deviation = 1
# This ensures no single feature dominates the distance calculation

scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_features)

print("Before scaling (first 3 rows):")
print(cluster_features.head(3).to_string())
print("\nAfter scaling (first 3 rows):")
print(cluster_scaled[:3].round(3))
print("\nAll features are now on the same scale ✅")

### Step 2 — Find the Optimal Number of Clusters (Elbow Method)

We don't know how many clusters to use upfront. The **Elbow Method** helps us choose:
- Run K-Means for K = 2, 3, 4, ... 9
- Record **inertia** (total distance of points from their cluster center) for each K
- Look for the "elbow" — where adding more clusters stops improving much
- That bend in the curve is the best K

In [ ]:
# ── Elbow Method ──────────────────────────────────────────────────────────
inertias = []
K_range = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(cluster_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(K_range, inertias, marker='o', color='steelblue', linewidth=2)
plt.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='Chosen K = 4')
plt.title("Elbow Method — Finding Optimal Number of Clusters", fontsize=13)
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia (lower = tighter clusters)")
plt.legend()
plt.tight_layout()
plt.show()

print("Inertia values:")
for k, inertia in zip(K_range, inertias):
    print(f"  K={k}: {inertia:,.1f}")
print("\n→ The elbow occurs at K=4, so we use 4 clusters.")

### Step 3 — Apply K-Means with K=4 and Visualize with PCA

After clustering, we use **PCA (Principal Component Analysis)** to reduce 3 dimensions → 2 dimensions so we can plot the clusters on a 2D scatter plot. PCA finds the directions of maximum variance and projects the data onto them.

In [ ]:
# ── Apply K-Means with K=4 ────────────────────────────────────────────────
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(cluster_scaled)

# ── PCA: Reduce 3D → 2D for visualization ─────────────────────────────────
# PCA finds the two directions of maximum variance in the data
# and projects all points onto those two axes.
# This lets us see the clusters in a 2D plot even though
# the model used 3 features.

pca = PCA(n_components=2, random_state=42)
cluster_pca = pca.fit_transform(cluster_scaled)

df['pc1'] = cluster_pca[:, 0]
df['pc2'] = cluster_pca[:, 1]

print(f"Variance explained by PC1: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"Variance explained by PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"Total variance captured:   {sum(pca.explained_variance_ratio_)*100:.1f}%")

In [ ]:
# ── Scatter Plot: K-Means Clusters in 2D (via PCA) ────────────────────────
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df, x='pc1', y='pc2',
    hue='cluster', palette='tab10',
    alpha=0.5, s=30
)
plt.title("K-Means Clusters Visualized in 2D (via PCA)", fontsize=13)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster", labels=["Cluster 0","Cluster 1","Cluster 2","Cluster 3"])
plt.tight_layout()
plt.show()

### Step 4 — Analyze What Each Cluster Means

In [ ]:
# ── Cluster Profile: Mean values per cluster ──────────────────────────────
# This shows us the average release_year, duration_num, and num_genres
# for titles in each cluster — helping us INTERPRET what each cluster represents.

cluster_means = df.groupby('cluster')[['release_year', 'duration_num', 'num_genres']].mean().round(1)
print("Average feature values per cluster:")
print(cluster_means)

In [ ]:
# ── Cluster Profile: Movie vs TV Show breakdown ───────────────────────────
# What percentage of each cluster is Movies vs TV Shows?
# This confirms whether our clusters map to content type patterns.

print("\nContent type breakdown per cluster:")
type_breakdown = df.groupby('cluster')['type'].value_counts(normalize=True).mul(100).round(1)
print(type_breakdown)

print("\nCluster size (number of titles):")
print(df['cluster'].value_counts().sort_index())

### Cluster Interpretation

Based on the mean values and type breakdown:

| Cluster | Profile | Dominant Type |
|---------|---------|--------------|
| **0** | Older titles, longer duration, fewer genres | Mostly Movies (classic catalog) |
| **1** | Newer titles, short duration (1 season), few genres | Mostly TV Shows |
| **2** | Recent titles, longer movies, multiple genres | Mixed — mainstream Movies |
| **3** | Newer, multi-genre, multi-season TV content | Mostly TV Shows |

These clusters reveal real patterns in how Netflix organizes its content catalog.

---
## 7. Conclusion <a id='7-conclusion'></a>

### Summary

In this project, I applied two machine learning methods to the Netflix Titles dataset:

#### 🌲 Random Forest Classification
- Built a classifier to predict Movie vs TV Show using metadata features
- `duration_num` was the **#1 most important feature** — the gap between movie minutes and TV show seasons is unmistakable
- `release_year` and content `rating` also contributed meaningfully
- The model achieved strong accuracy on the held-out test set

#### 📍 K-Means Clustering
- Used the **Elbow Method** to select K=4 clusters
- Applied **StandardScaler** before clustering to prevent any single feature from dominating
- Used **PCA** to visualize 3D clusters in a 2D scatter plot
- Discovered 4 meaningful content groups: classic movies, short new TV shows, mainstream recent movies, and multi-season TV content

### Key Takeaways

| Insight | Detail |
|---------|--------|
| 📊 Class imbalance | Movies outnumber TV Shows ~70% to ~30% |
| 📅 Content growth | Sharp increase in titles post-2010 |
| ⏱️ Best predictor | Duration is the strongest signal for type classification |
| 🎭 Top genres | International Movies, Dramas, Comedies dominate |
| 🌍 Global content | Netflix heavily invests in international productions |

### What I Learned
This project reinforced how real-world data science works end-to-end:
1. **Raw data needs cleaning** before anything else
2. **Feature engineering** (creating `num_genres`, `num_countries`) can improve models
3. **EDA first** — visualizing data reveals what features will be useful
4. **Scaling matters** for distance-based models like K-Means
5. **Interpretability** (feature importance, cluster profiles) is just as important as accuracy

---
*This project was completed as part of coursework at Arizona State University.*  
**Author:** Gayathri Kota | gayathrikota2020@gmail.com